# Veto Variants Testing

Compare routing veto strategies on a small sample without modifying
pipeline nodes.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

from taxomind.pipelines.inference.nodes import (
    build_retrieval_index,
    load_taxonomy_graph,
    prepare_scoring_views,
    retrieve_candidates,
    compute_multiview_score,
)

from sentence_transformers import SentenceTransformer

project_path = Path.cwd()
if not (project_path / "conf").exists():
    project_path = project_path.parent
bootstrap_project(project_path)


In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)
for name in ["kedro", "taxomind", "sentence_transformers", "transformers", "urllib3"]:
    logging.getLogger(name).setLevel(logging.ERROR)


## Load targets and sample


In [ ]:
SAMPLE_PER_TAXONOMY = 2000
DATASET_NAME = "classifai_validation_data"
TAXONOMY_KEY = "ISCO"

with KedroSession.create(project_path=project_path) as session:
    run_result = session.run(pipeline_name="error_analysis")

def _unwrap(value):
    return value.load() if hasattr(value, "load") else value

classifai_targets = _unwrap(run_result.get("error_analysis_classifai_targets"))
taxonomy_training_targets = _unwrap(run_result.get("error_analysis_taxonomy_training_targets"))
training_sentences_targets = _unwrap(run_result.get("error_analysis_training_sentences_targets"))

datasets = {
    "classifai_validation_data": classifai_targets,
    "taxonomy_training": taxonomy_training_targets,
    "training_sentences": training_sentences_targets,
}

targets = datasets[DATASET_NAME]
targets = targets[targets["taxonomy_key"] == TAXONOMY_KEY].copy()
if len(targets) > SAMPLE_PER_TAXONOMY:
    targets = targets.sample(n=SAMPLE_PER_TAXONOMY, random_state=7)
targets = targets.reset_index(drop=True)
targets["query_id"] = targets.index
targets.head()


## Build inference assets


In [ ]:
with KedroSession.create(project_path=project_path) as session:
    context = session.load_context()
    params = context.params
    partitions = context.catalog.load("taxonomy_index")

taxonomy_df = partitions[TAXONOMY_KEY]()
taxonomy_graph = load_taxonomy_graph(taxonomy_df)
retrieval_index = build_retrieval_index(taxonomy_df)
scoring_views = prepare_scoring_views(taxonomy_df)

level_map = {str(row["code"]).strip(): int(row["level"]) for _, row in taxonomy_df.iterrows()}
parent_map = {
    str(row["code"]).strip(): ("__root__" if pd.isna(row["parentCode"]) or str(row["parentCode"]).strip() == "" else str(row["parentCode"]).strip())
    for _, row in taxonomy_df.iterrows()
}

model_name = params.get("model_name")
embedding_model = SentenceTransformer(model_name, trust_remote_code=True)

query_texts = targets["query_text"].fillna("").astype(str).tolist()
query_embeddings = embedding_model.encode(
    query_texts,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
)


## Routing variants


In [ ]:
def is_ancestor(ancestor: str, code: str, parent_map: dict) -> bool:
    if not ancestor or not code or ancestor == "__root__":
        return False
    current = code
    while current and current != "__root__":
        if current == ancestor:
            return True
        current = parent_map.get(current)
    return False


def _route_from_root(
    query_embedding,
    query_text,
    candidates_dict,
    scoring_views,
    taxonomy_graph,
    level_map,
    variant,
    params,
    beam_root=None,
):
    V_codes = set(candidates_dict["V_codes"])
    if beam_root:
        current_parent = beam_root
        current_level = level_map.get(beam_root, 1)
    else:
        current_parent = "__root__"
        current_level = 0

    while True:
        all_children = taxonomy_graph.get(current_parent, [])
        if not all_children:
            return current_parent, current_level, 0.0, False, "leaf_node_reached"

        candidate_children = [c for c in all_children if c in V_codes]
        if not candidate_children:
            return current_parent, current_level, 0.0, True, "no_candidate_children"

        scores = {}
        for child in candidate_children:
            scores[child] = compute_multiview_score(
                query_embedding=query_embedding,
                query_text=query_text,
                node_code=child,
                scoring_views=scoring_views,
                evidence_tau=params["evidence_tau"],
                evidence_max_beta=params["evidence_max_beta"],
                short_query_tokens=params["short_query_tokens"],
            )
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        best_child, best_score = ranked[0]
        second_score = ranked[1][1] if len(ranked) > 1 else -1.0
        margin = best_score - second_score

        if current_parent == "__root__":
            parent_score = -1.0
        else:
            parent_score = compute_multiview_score(
                query_embedding=query_embedding,
                query_text=query_text,
                node_code=current_parent,
                scoring_views=scoring_views,
                evidence_tau=params["evidence_tau"],
                evidence_max_beta=params["evidence_max_beta"],
                short_query_tokens=params["short_query_tokens"],
            )

        dispersion = float(np.std(list(scores.values()))) if len(scores) > 1 else 0.0

        stop_for_margin = margin < params["min_descent_gap"]

        if variant == "no_veto":
            parent_veto = False
        elif variant == "fixed_veto":
            parent_veto = parent_score + params["parent_veto_margin"] >= best_score
        elif variant == "depth_decay":
            decay_margin = params["parent_veto_margin"] * (params["veto_decay"] ** max(current_level - 1, 0))
            parent_veto = parent_score + decay_margin >= best_score
        elif variant == "conditional_veto":
            parent_veto = parent_score + params["parent_veto_margin"] >= best_score
        elif variant == "relative_gap":
            rel_gap = (best_score - parent_score) / max(abs(best_score), 1e-6)
            parent_veto = rel_gap < params["relative_gap_threshold"]
        elif variant == "dispersion_aware":
            if dispersion < params["dispersion_threshold"]:
                parent_veto = parent_score + params["parent_veto_margin"] >= best_score
            else:
                parent_veto = False
        else:
            parent_veto = False

        if variant == "conditional_veto":
            should_stop = stop_for_margin and parent_veto
        else:
            should_stop = stop_for_margin or parent_veto

        if should_stop:
            stopping_reason = "margin" if stop_for_margin else "parent_veto"
            return current_parent, current_level, parent_score, stop_for_margin, stopping_reason

        current_parent = best_child
        current_level = level_map.get(best_child, current_level + 1)


def route_variant(
    query_embedding,
    query_text,
    candidates_dict,
    scoring_views,
    taxonomy_graph,
    level_map,
    variant,
    params,
):
    beam_roots = candidates_dict.get("beam_roots") or [None]
    best = None
    for root in beam_roots:
        code, level, score, ambiguous, reason = _route_from_root(
            query_embedding,
            query_text,
            candidates_dict,
            scoring_views,
            taxonomy_graph,
            level_map,
            variant,
            params,
            beam_root=root,
        )
        final_score = score
        if code and code != "__root__":
            final_score = compute_multiview_score(
                query_embedding=query_embedding,
                query_text=query_text,
                node_code=code,
                scoring_views=scoring_views,
                evidence_tau=params["evidence_tau"],
                evidence_max_beta=params["evidence_max_beta"],
                short_query_tokens=params["short_query_tokens"],
            )
        candidate = {
            "predicted_code": code,
            "predicted_level": level,
            "score": final_score,
            "ambiguous": ambiguous,
            "stopping_reason": reason,
        }
        if best is None or candidate["score"] > best["score"]:
            best = candidate
    return best


## Run variants


In [ ]:
variant_params = {
    "min_descent_gap": float(params.get("inference", {}).get("min_descent_gap", 0.05)),
    "parent_veto_margin": float(params.get("inference", {}).get("parent_veto_margin", 0.05)),
    "evidence_tau": float(params.get("inference", {}).get("evidence_tau", 10.0)),
    "evidence_max_beta": float(params.get("inference", {}).get("evidence_max_beta", 0.8)),
    "short_query_tokens": int(params.get("inference", {}).get("short_query_tokens", 2)),
    "veto_decay": 0.7,
    "relative_gap_threshold": 0.05,
    "dispersion_threshold": 0.05,
}

variants = [
    "fixed_veto",
    "no_veto",
    "depth_decay",
    "conditional_veto",
    "relative_gap",
    "dispersion_aware",
]

pred_rows = []
for idx, row in targets.iterrows():
    query_text = row["query_text"]
    query_embedding = query_embeddings[idx]
    candidates_dict = retrieve_candidates(
        query_embedding=query_embedding,
        retrieval_index=retrieval_index,
        retrieval_k=int(params.get("inference", {}).get("retrieval_k", 20)),
        beam_count=int(params.get("inference", {}).get("beam_count", 2)),
    )
    for variant in variants:
        result = route_variant(
            query_embedding=query_embedding,
            query_text=query_text,
            candidates_dict=candidates_dict,
            scoring_views=scoring_views,
            taxonomy_graph=taxonomy_graph,
            level_map=level_map,
            variant=variant,
            params=variant_params,
        )
        pred_rows.append({
            "query_id": row["query_id"],
            "variant": variant,
            **result,
        })

preds = pd.DataFrame(pred_rows)
preds.head()


## Evaluate variants


In [ ]:
def evaluate_variant(preds_df, targets_df, parent_map):
    merged = preds_df.merge(
        targets_df[["query_id", "target_code", "target_level"]],
        on="query_id",
        how="left",
    )
    valid = merged[merged["target_code"].fillna("").str.strip() != ""].copy()
    valid["exact_match"] = valid["predicted_code"] == valid["target_code"]
    valid["level_match"] = valid["predicted_level"] == valid["target_level"]
    valid["under_spec"] = valid.apply(
        lambda r: is_ancestor(r["predicted_code"], r["target_code"], parent_map)
        and r["predicted_code"] != r["target_code"],
        axis=1,
    )
    valid["over_spec"] = valid.apply(
        lambda r: is_ancestor(r["target_code"], r["predicted_code"], parent_map)
        and r["predicted_code"] != r["target_code"],
        axis=1,
    )
    valid["ancestor_match"] = valid.apply(
        lambda r: is_ancestor(r["predicted_code"], r["target_code"], parent_map)
        or r["predicted_code"] == r["target_code"],
        axis=1,
    )
    valid["wrong_branch"] = ~(valid["under_spec"] | valid["over_spec"] | valid["exact_match"])

    summary = valid.groupby("variant").agg(
        rows=("query_id", "size"),
        exact_match_rate=("exact_match", "mean"),
        level_match_rate=("level_match", "mean"),
        ancestor_match_rate=("ancestor_match", "mean"),
        under_spec_rate=("under_spec", "mean"),
        over_spec_rate=("over_spec", "mean"),
        wrong_branch_rate=("wrong_branch", "mean"),
    ).reset_index()

    return valid, summary

eval_df, summary_df = evaluate_variant(preds, targets, parent_map)
summary_df


## Charts


In [ ]:
if not summary_df.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(summary_df))
    ax.bar(x, summary_df["exact_match_rate"], label="exact_match")
    ax.bar(x, summary_df["ancestor_match_rate"], bottom=summary_df["exact_match_rate"], label="ancestor_match")
    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["variant"], rotation=45, ha="right")
    ax.set_ylim(0, 1)
    ax.set_title("Match rates by variant")
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
summary_df

In [ ]:
summary_df_isco = summary_df.copy()
summary_df_isco